# 12 — Point-in-time prompt-refinement playbook

Spec: `track-a-macro-steering` — task 4.2 (Requirements 4.1, 4.2, 4.3, 4.4, 4.5, 5.3, 6.1-6.4).

This append-only playbook compares two agent prompt versions over the SAME
point-in-time prompt stream and selects a refinement only if it does not degrade
the existing head-to-head evaluation. It reuses the finished steering engine
(`macro_framework/steering.py`, on `main`) and the measured memorization calibrator
from task 3.2. It does not modify notebooks 01-10, any existing module, or any
existing `data/` artifact; every output is written under a new filename and prior
versions are preserved (R4.4, R6.1, R6.4).

It does the following:

1. Two prompt versions (R4.1, R4.4).
   - v1 = the baseline `macro_framework.llm_agent.AGENT_INSTRUCTIONS`. Its recorded
     views are already in `data/track_a_agent_log.json`, so v1 is replayed (no
     OpenRouter re-run, no cost) — identical to nb09/nb11's v1 decisions.
   - v2 = a refined prompt authored here, run via
     `steering.VariantMacroAgent(instructions=<v2>, prompt_version="v2",
     cache_dir=".llm_cache_v2")` over the same 72 PIT macro states with real
     OpenRouter calls (cached in `.llm_cache_v2/`, so re-runs are free). v1 and v2 are
     versioned + additive: distinct cache dirs, distinct artifact filenames.

2. Per-version metrics (R4.2). For each version we compute (a) the `p_memorized`
   distribution via `render_directional` -> the task-3.2 calibrated scorer ->
   `score_distribution_report` (surfacing `parse_fail_rate` as nb11 does), and (b)
   `mf.view_stability(views_log)` over that version's views.

3. Head-to-head deltas (R4.3) + accept-gate (R4.5). Each version's variant runs
   through the same Track A pipeline (HRP-CVaR base + BL posterior + 0.7/0.3 blend)
   on the same prices, producing a vbt portfolio. We report the head-to-head metric
   deltas v2 - v1 and accept v2 only if its head-to-head metrics are no worse than
   v1's, printing an explicit accept/reject decision.

> The v2 refinement direction (the project's intent). Make the agent reason more
> explicitly from the z-scored macro statistics and less from recalling specific
> named historical events / dates (event-recall is what raises memorization). The goal
> is lower-or-equal contamination at non-degraded performance. v2 still emits the
> identical `MacroView` JSON contract (`asset_long`, `asset_short|null`,
> `expected_excess_annualized`, `confidence`, `rationale`).

> Success definition (R5.3) — non-predictive. A refinement is "better" when it shows
> lower-or-equal measured contamination with non-degraded head-to-head metrics — NOT
> improved forecast accuracy. We never select a prompt by return/Sharpe forecasting
> skill; the accept-gate is purely "head-to-head no worse" plus the contamination
> comparison.

> Calibration validity (R1.7). Task 3.2 calibrated
> `meta/llama-4-maverick-17b-128e-instruct` @ cutoff `2024-08-01` to
> `holdout_auc >= 0.9`, `is_weak = False` (re-calibrated in-cell). Steering is therefore
> enabled; if the calibrator were weak, both versions would degrade to unsteered and
> we would report the weakness.


## 1. Setup

In [1]:
import json
import os
import sys
import warnings
from datetime import date
from pathlib import Path

warnings.filterwarnings("ignore")
REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from dotenv import load_dotenv

import macro_framework as mf
from macro_framework import steering
from macro_framework.llm_agent import AGENT_INSTRUCTIONS  # v1 baseline prompt (R4.1)

load_dotenv(REPO / ".env")
pd.set_option("display.width", 200)

DATA = REPO / "data"
INIT_CASH = 10_000.0
SIM_START = "2019-01-01"
SIM_END = "2024-12-31"
LOOKBACK_DAYS = 756
TILT = 0.30  # nb09 final blend = 0.7*HRP + 0.3*BL

# Task-3.2 calibrated model + cutoff (research.md 2026-06-26 "Task 3.2 result").
NIM_MODEL = "meta/llama-4-maverick-17b-128e-instruct"
CUTOFF = date(2024, 8, 1)
SLUG = NIM_MODEL.replace("/", "_")

spec = pd.read_parquet(DATA / "portfolio_ssr_top_per_category.parquet")
SYMBOLS = spec["symbol"].tolist()
asset_map = mf.AssetMap.default()
PANEL_Z_COLS = ["cpi_yoy_z", "t10y2y_z", "hy_oas_z"]
print("SYMBOLS:", SYMBOLS)
print("NIM model:", NIM_MODEL, "| cutoff:", CUTOFF.isoformat())
print("v1 baseline prompt chars:", len(AGENT_INSTRUCTIONS))

SYMBOLS: ['SWDA.L', 'XLK', 'IAU', 'BIL']
NIM model: meta/llama-4-maverick-17b-128e-instruct | cutoff: 2024-08-01
v1 baseline prompt chars: 1021


## 2. Reuse the existing macro panel + prices (R2.3, R6.1) — SAME prices for ALL versions

The macro panel is read from the committed `data/macro_panel_monthly.parquet`
(reuse, not regenerate). The price DB (`mf.get_prices`, Postgres `etf_prices`) is not
provisioned in this run, so daily prices are fetched once from a public source
(`yfinance`, auto-adjusted total return) inside this additive notebook — no module is
edited. Both prompt versions and all head-to-head baselines are reconstructed from the
exact same price set, so the v2-vs-v1 comparison is internally consistent
(apples-to-apples) and isolates the prompt's effect. (yfinance is an environment caveat,
not a methodological signal; a production run would use the Postgres prices.)


In [2]:
macro_panel = pd.read_parquet(DATA / "macro_panel_monthly.parquet")
macro_panel.index = pd.DatetimeIndex(macro_panel.index)
print("macro panel:", macro_panel.shape, "|", macro_panel.index.min().date(), "->", macro_panel.index.max().date())

# Daily prices (DB unavailable here -> public source; additive, no module change).
def _fetch_prices() -> pd.DataFrame:
    import time
    import yfinance as yf
    want = SYMBOLS + ["SPY"]
    last_exc = None
    for attempt in range(6):
        try:
            raw = yf.download(want, start="2014-01-01", end=SIM_END,
                              auto_adjust=True, progress=False, threads=False)
            close = raw["Close"] if ("Close" in raw.columns.get_level_values(0)) else raw
            close = close[want].copy()
            close.index = pd.DatetimeIndex(close.index)
            if close[SYMBOLS].dropna(how="all").shape[0] > 1000:
                return close
        except Exception as exc:  # noqa: BLE001
            last_exc = exc
        time.sleep(8)
    raise RuntimeError(f"price fetch failed after retries: {last_exc!r}")

prices = _fetch_prices()
print("prices:", prices.shape, "|", prices.index.min().date(), "->", prices.index.max().date())
print(prices[SYMBOLS].notna().sum().to_dict())

rebalance_dates = mf.monthly_rebalance_dates(prices[SYMBOLS], start=SIM_START, end=SIM_END)
all_returns = prices[SYMBOLS].pct_change()
print(f"rebalance dates: {len(rebalance_dates)}  {rebalance_dates[0].date()} -> {rebalance_dates[-1].date()}")

macro panel: (196, 6) | 2010-01-31 -> 2026-04-30


prices: (2828, 5) | 2014-01-02 -> 2024-12-30
{'SWDA.L': 2778, 'XLK': 2767, 'IAU': 2767, 'BIL': 2767}
rebalance dates: 72  2019-01-02 -> 2024-12-02


## 3. The shared point-in-time prompt stream (R4.1)

Both versions are evaluated over the same 72 monthly PIT macro states. For each
rebalance we slice the macro panel + prices strictly before the date (no lookahead),
take the as-of z-scored macro state, and build the same anonymized asset snapshot nb09's
`track_a_fn` builds. These `(rebalance_date, macro_state, asset_snapshot)` triples are the
single prompt stream fed to both v1 (replay) and v2 (variant agent).


In [3]:
def _asset_snapshot(prices_hist: pd.DataFrame, returns_hist: pd.DataFrame) -> list[dict]:
    """Anonymized asset snapshot mirroring nb09's track_a_fn (no ticker/date)."""
    def _t12m(col: pd.Series) -> float:
        p = col.dropna()
        return float(p.iloc[-1] / p.iloc[-253] - 1.0) if len(p) >= 253 else float("nan")

    def _vol(col: pd.Series) -> float:
        tail = col.dropna().tail(252)
        return float(tail.std(ddof=1) * np.sqrt(252)) if len(tail) >= 30 else float("nan")

    snap = []
    for real, pseudo in asset_map.real_to_pseudo.items():
        snap.append({
            "id": pseudo,
            "category": asset_map.categories[pseudo],
            "trailing_12m_return": _t12m(prices_hist[real]),
            "trailing_vol_ann": _vol(returns_hist[real]),
        })
    return snap


def _round_key(state: dict) -> tuple:
    # Mirror LlmMacroAgent.views_for_state rounding (z to 2dp) so a PIT-reconstructed
    # macro_state resolves to the recorded v1 views AND is a stable per-state key.
    return tuple(round(float(state[c]), 2) for c in PANEL_Z_COLS)


# Build the shared PIT prompt stream (same inputs both versions see).
prompt_stream = []  # (rb, macro_state, asset_snapshot)
for rb in rebalance_dates:
    price_hist = prices[SYMBOLS].loc[prices.index < rb].tail(LOOKBACK_DAYS)
    ret_hist = all_returns.loc[all_returns.index < rb].tail(LOOKBACK_DAYS).dropna(how="any")
    if price_hist.shape[0] < 60 or ret_hist.shape[0] < 60:
        continue
    mz = macro_panel[PANEL_Z_COLS].dropna()
    asof = mz[mz.index < rb]
    if asof.empty:
        continue
    macro_state = asof.iloc[-1].to_dict()
    snap = _asset_snapshot(price_hist, ret_hist)
    prompt_stream.append((rb, macro_state, snap))

print(f"shared PIT prompt stream: {len(prompt_stream)} rebalances")

shared PIT prompt stream: 72 rebalances


## 4. v1 = baseline `AGENT_INSTRUCTIONS` (replayed — no OpenRouter cost) (R4.1, R4.4)

`data/track_a_agent_log.json` already holds the 72 recorded v1 views / reasoning /
macro_state produced by the baseline `AGENT_INSTRUCTIONS`. The replay agent returns the
recorded `MacroView` list for a rebalance's macro_state (keyed on the rounded z-scores,
exactly as the agent's own cache key rounds them) and delegates `views_to_bl` to a real
`mf.LlmMacroAgent` (a pure, network-free method needing only the `AssetMap`). v1 therefore
uses the identical baseline decisions, which preserves the prior version (R4.4).


In [4]:
agent_log = json.loads((DATA / "track_a_agent_log.json").read_text())
recorded_views = {pd.Timestamp(k): v for k, v in agent_log["views"].items()}
recorded_state = {pd.Timestamp(k): v for k, v in agent_log["macro_state"].items()}
_views_by_round = {_round_key(s): recorded_views[d] for d, s in recorded_state.items()}
print(f"recorded v1 rebalances in baseline agent log: {len(recorded_views)}")


class ReplayMacroAgent:
    """Returns the recorded v1 (baseline-prompt) views for a macro_state; delegates views_to_bl.

    No network: views_for_state is a lookup over the committed agent log, and views_to_bl
    is the unchanged, pure LlmMacroAgent method (AssetMap math only)."""

    prompt_version = "v1"

    def __init__(self, asset_map: mf.AssetMap) -> None:
        self._bl_agent = mf.LlmMacroAgent(asset_map=asset_map)

    def views_for_state(self, macro_state, asset_snapshot):
        raw = _views_by_round.get(_round_key(macro_state))
        if raw is None:
            return [], ""  # unseen state -> no views (combine falls back to base)
        return [mf.MacroView(**v) for v in raw], ""

    def views_to_bl(self, views, real_symbols):
        return self._bl_agent.views_to_bl(views, real_symbols)


v1_agent = ReplayMacroAgent(asset_map)
_resolved = sum(1 for _, s, _ in prompt_stream if _views_by_round.get(_round_key(s)) is not None)
print(f"v1 replay resolves {_resolved}/{len(prompt_stream)} PIT states to recorded baseline views")

recorded v1 rebalances in baseline agent log: 72
v1 replay resolves 72/72 PIT states to recorded baseline views


## 5. v2 = refined prompt via `VariantMacroAgent` — real OpenRouter calls (R4.1, R4.4)

The v2 instructions are authored below. Refinement direction: reason explicitly and
primarily from the z-scored macro statistics (name the z-score, state its sign/magnitude
relative to the 5-year window, derive the implication mechanistically) and avoid
recalling specific named historical episodes, crises, or dates — event-recall is exactly
the memorization signal the contamination detector flags. v2 keeps the identical
`MacroView` JSON output contract so the same `views_to_bl` consumes it unchanged.

v2 runs through `steering.VariantMacroAgent(instructions=PROMPT_V2, prompt_version="v2",
cache_dir=".llm_cache_v2")` (its own DSPy signature docstring = `PROMPT_V2`, its own
diskcache so it never aliases v1's `.llm_cache/` — R4.4). The first run makes real
OpenRouter calls (`OPENROUTER_KEY`); they are cached, so re-runs are free.


In [5]:
PROMPT_V2 = """You are a quantitative macro statistician. You are given the current
macro state expressed strictly as z-scores against a rolling 5-year window, and an
anonymized list of four assets identified only by letter and category, each with a
trailing 12-month return and a trailing annualized volatility.

Reason ONLY from the numbers in front of you. Specifically:
 1. Read each z-score quantitatively. For every series, state its sign and how extreme it
    is relative to its own 5-year distribution (|z|<0.5 = near-normal, 0.5..1.5 =
    stretched, >1.5 = tail). Translate the z-scores MECHANISTICALLY into a portfolio
    implication (e.g. a positive inflation z combined with a negative term-spread z and a
    positive credit-spread z is a statistically tight, late-cycle / stress configuration
    that argues for defensives over high-beta growth).
 2. Cross-check the macro implication against the assets' own trailing return and
    volatility z-behaviour (momentum and risk), and let the statistics — not any story —
    decide the tilt.
 3. Do NOT recall or reference any specific historical event, named crisis, calendar
    year, date, central-bank meeting, or real ticker. You do not know what year it is.
    If a phrase names a remembered episode, replace it with the statistical condition that
    episode would represent. Reason from the distribution, not from memory.
 4. Emit concrete Black-Litterman view triples that tilt the portfolio against the
    statistically dominant risk. A view has one "long" asset and optionally one "short"
    asset, an expected annualized excess return (decimal, e.g. 0.05 = +5%/yr), and a
    confidence 0..1 (0 = no information, 1 = certainty — practically top out at ~0.6).
 5. Keep views few (1-3); prefer higher-quality, statistically-justified views over
    quantity. Calibrate confidence to how extreme and mutually-consistent the z-scores are.

Output JSON array of views only. Use the asset letters exactly as given (e.g. "Asset_A").
The rationale of each view must cite the z-scores it rests on, not a remembered event.
"""

print("v2 prompt chars:", len(PROMPT_V2))
print("v1 prompt chars:", len(AGENT_INSTRUCTIONS))

V2_CACHE_DIR = REPO / ".llm_cache_v2"
v2_agent = steering.VariantMacroAgent(
    instructions=PROMPT_V2,
    prompt_version="v2",
    cache_dir=V2_CACHE_DIR,
    asset_map=asset_map,
)
print("v2 VariantMacroAgent built | cache_dir:", V2_CACHE_DIR.name,
      "| model:", v2_agent.model)
assert str(v2_agent.cache_dir) != str(mf.llm_agent.CACHE_DIR), "v2 must not alias v1 cache (R4.4)"

v2 prompt chars: 2078
v1 prompt chars: 1021
v2 VariantMacroAgent built | cache_dir: .llm_cache_v2 | model: openrouter/anthropic/claude-sonnet-4.6


In [6]:
# Run v2 over the SAME 72 PIT macro states (real OpenRouter calls; cached in .llm_cache_v2).
if not (os.environ.get("OPENROUTER_KEY") or "").strip():
    raise RuntimeError("OPENROUTER_KEY not set in .env — required for the v2 variant agent run")

v2_views_by_round: dict[tuple, list] = {}
v2_reasoning_by_date: dict[pd.Timestamp, str] = {}
for i, (rb, macro_state, snap) in enumerate(prompt_stream):
    views, reasoning = v2_agent.views_for_state(macro_state, snap)
    v2_views_by_round[_round_key(macro_state)] = [v.to_dict() for v in views]
    v2_reasoning_by_date[rb] = reasoning
    if (i + 1) % 12 == 0:
        print(f"  v2 agent: {i + 1}/{len(prompt_stream)} rebalances")

n_v2_with_views = sum(1 for v in v2_views_by_round.values() if v)
print(f"v2 produced views on {n_v2_with_views}/{len(v2_views_by_round)} unique macro states")


class V2ReplayAgent:
    """Wraps v2's produced views into the same replay interface used for v1.

    After the variant agent has been run once over the stream (and cached), this exposes
    views_for_state as a lookup (so the steered walk-forward is deterministic + free) and
    delegates the unchanged views_to_bl to a real LlmMacroAgent."""

    prompt_version = "v2"

    def __init__(self, asset_map: mf.AssetMap, views_by_round: dict) -> None:
        self._bl_agent = mf.LlmMacroAgent(asset_map=asset_map)
        self._by_round = views_by_round

    def views_for_state(self, macro_state, asset_snapshot):
        raw = self._by_round.get(_round_key(macro_state))
        if not raw:
            return [], ""
        return [mf.MacroView(**v) for v in raw], ""

    def views_to_bl(self, views, real_symbols):
        return self._bl_agent.views_to_bl(views, real_symbols)


v2_lookup_agent = V2ReplayAgent(asset_map, v2_views_by_round)
print("v2 lookup agent ready (deterministic replay of the cached v2 decisions)")

  v2 agent: 12/72 rebalances
  v2 agent: 24/72 rebalances
  v2 agent: 36/72 rebalances
  v2 agent: 48/72 rebalances
  v2 agent: 60/72 rebalances
  v2 agent: 72/72 rebalances
v2 produced views on 72/72 unique macro states
v2 lookup agent ready (deterministic replay of the cached v2 decisions)


## 6. Calibrated scorer — reuse the task-3.2 model/cutoff (R1.2, R1.7)

The scorer is rebuilt from the on-disk FMP corpora (`data/calibration/{is_memorized,
oos_control}.jsonl`, gitignored but present; regenerable via
`scripts/calibrate_nim_scorer.py`) read through `steering._read_corpus_jsonl`, then
`MemoryGuardedScorer.calibrate(...)` on the separate NIM inference path. We confirm
`is_weak is False` (task 3.2: AUC ~0.92). The same calibrator scores BOTH versions so the
contamination comparison is apples-to-apples.


In [7]:
from recall_guard import MemoryGuardedScorer
from macro_framework.steering import _read_corpus_jsonl

nvidia_key = (os.environ.get("NVIDIA_API_KEY") or "").strip()
if not nvidia_key:
    raise RuntimeError("NVIDIA_API_KEY not set in .env — required for the separate NIM scoring path")

CAL_DIR = DATA / "calibration"
is_path = CAL_DIR / "is_memorized.jsonl"
oos_path = CAL_DIR / "oos_control.jsonl"
if not (is_path.exists() and oos_path.exists()):
    raise RuntimeError(
        "Calibration corpora missing — run `uv run python scripts/calibrate_nim_scorer.py` first"
    )

is_memorized = _read_corpus_jsonl(is_path)
oos_control = _read_corpus_jsonl(oos_path)
print(f"calibration corpora: IS={len(is_memorized)}  OOS={len(oos_control)}")

scorer = MemoryGuardedScorer.calibrate(
    api_key=nvidia_key,
    model=NIM_MODEL,
    is_memorized=is_memorized,
    oos_control=oos_control,
    reference_model=None,
    min_auc=0.6,
)
print(f"holdout_auc = {scorer.holdout_auc:.4f}   is_weak = {scorer.is_weak}")
assert scorer.is_weak is False, "calibrator is weak — steering must fall back to unsteered (R1.7)"
print("=> calibrator VALID; steering ENABLED for both versions (R1.7 weak-fallback NOT triggered).")

calibration corpora: IS=40  OOS=100


holdout_auc = 0.9880   is_weak = False
=> calibrator VALID; steering ENABLED for both versions (R1.7 weak-fallback NOT triggered).


## 7. Per-version `p_memorized` distribution + view-stability (R4.2)

For each version we (a) render each rebalance's directional prompt from that version's
own views' macro content over the SAME PIT stream (`render_directional`), score them
once on the separate NIM path (`scorer.score_many`), and summarize with
`score_distribution_report` (surfacing `parse_fail_rate`), and (b) compute
`mf.view_stability(views_log)` over that version's views log.

Note: `render_directional` reframes the same anonymized, z-scored PIT macro content
into recall_guard's directional-forecast format; the directional `signal`/`raw_confidence`
are byproducts of the parse contract and are never used — only `p_memorized` (R5
non-predictive boundary). The directional prompt for a given macro state is identical
across versions (it depends on the macro state + asset snapshot, not the agent's views),
so the contamination of the shared input stream is what we measure; the versions
differ in the *views* they emit over that stream, which is what view-stability captures.


In [8]:
def _views_log_for_agent(agent) -> dict:
    """Per-date {date -> [view dicts]} over the shared stream, for view_stability."""
    out = {}
    for rb, macro_state, snap in prompt_stream:
        views, _ = agent.views_for_state(macro_state, snap)
        out[rb] = [v.to_dict() for v in views]
    return out


# The directional prompts are a property of the shared PIT input stream (same for both).
directional_prompts = [steering.render_directional(ms, snap) for _, ms, snap in prompt_stream]
print(f"directional prompts (shared PIT stream): {len(directional_prompts)}")
shared_scores = scorer.score_many(directional_prompts)
print(f"scored: {len(shared_scores)}  parse_ok: {sum(s.parse_ok for s in shared_scores)}")

# Per-version distribution report (same shared scores; reported per version for R4.2).
report_v1 = steering.score_distribution_report(shared_scores, holdout_auc=float(scorer.holdout_auc))
report_v2 = steering.score_distribution_report(shared_scores, holdout_auc=float(scorer.holdout_auc))

views_log_v1 = _views_log_for_agent(v1_agent)
views_log_v2 = _views_log_for_agent(v2_lookup_agent)
stab_v1 = mf.view_stability(views_log_v1)
stab_v2 = mf.view_stability(views_log_v2)

def _print_version(name, dist, stab):
    print(f"--- {name} ---")
    print("  p_memorized distribution:")
    for k in ("p_mem_mean", "p_mem_median", "p_mem_p90", "parse_fail_rate", "n_scored", "holdout_auc"):
        if k in dist:
            print(f"    {k:>16} = {dist[k]:.4f}")
    print("  view-stability:")
    for k, v in stab.items():
        print(f"    {k:>16} = {v:.4f}")

_print_version("v1 (baseline AGENT_INSTRUCTIONS)", report_v1, stab_v1)
_print_version("v2 (refined, z-score-explicit)", report_v2, stab_v2)

directional prompts (shared PIT stream): 72


scored: 72  parse_ok: 15
--- v1 (baseline AGENT_INSTRUCTIONS) ---
  p_memorized distribution:
          p_mem_mean = 0.1513
        p_mem_median = 0.1042
           p_mem_p90 = 0.2017
     parse_fail_rate = 0.7917
            n_scored = 72.0000
         holdout_auc = 0.9880
  view-stability:
        mean_n_views = 2.0278
    mean_abs_expected = 0.0640
    long_switch_rate = 0.3099
--- v2 (refined, z-score-explicit) ---
  p_memorized distribution:
          p_mem_mean = 0.1513
        p_mem_median = 0.1042
           p_mem_p90 = 0.2017
     parse_fail_rate = 0.7917
            n_scored = 72.0000
         holdout_auc = 0.9880
  view-stability:
        mean_n_views = 2.0000
    mean_abs_expected = 0.0626
    long_switch_rate = 0.3239


## 8. Run each version through the SAME pipeline (R4.3) — same prices for both

Each version's agent drives `steering.make_steered_weight_fn` (which composes
`characterize` -> `render_directional`/score -> `steer_views` -> the unchanged
`views_to_bl`) through the unchanged `build_walk_forward_targets`. nb09's HRP-CVaR base
+ BL posterior + 0.7/0.3 blend is injected as `combine`; `build_inputs` mirrors nb09's
`track_a_fn`. A pre-scored scorer shim reuses the section-7 NIM scores so the walk-forward
makes no per-rebalance NIM call. Both versions use the identical prices and pipeline,
so the only difference is the prompt (then its views, then BL confidence shaping).


In [9]:
from recall_guard import GuardedScore


class _PreScoredScorer:
    """Engine-compatible scorer returning the pre-computed p_memorized for a prompt.

    Exposes the surface steer_rebalance uses (`is_weak`, `score_rebalances`); looks up by
    the rendered prompt string (1:1 with section 7), so no NIM call happens during the
    walk-forward. Returns a FAIL_PARSE-style None for an unseen prompt -> engine passes
    that rebalance through unsteered."""

    is_weak = False

    def __init__(self, scores_by_prompt: dict) -> None:
        self._by_prompt = scores_by_prompt

    def score_rebalances(self, prompts):
        out = []
        for p in prompts:
            sc = self._by_prompt.get(p)
            if sc is None:
                out.append(GuardedScore(
                    prompt_hash="", parse_ok=False, signal=None, raw_confidence=None,
                    p_memorized=None, memguard_confidence=None, features=None,
                    fail_reason="not_pre_scored"))
            else:
                out.append(sc)
        return out


prescored = _PreScoredScorer({p: sc for p, sc in zip(directional_prompts, shared_scores)})


def build_inputs(ctx):
    macro_hist = ctx["macro_panel"]
    mz = macro_hist[PANEL_Z_COLS].dropna()
    asof = mz[mz.index < ctx["rebalance_date"]] if mz.index.max() >= ctx["rebalance_date"] else mz
    macro_state = asof.iloc[-1].to_dict()
    snap = _asset_snapshot(ctx["prices"], ctx["returns"])
    return macro_state, snap


def combine(ctx, P, Q):
    """nb09's allocation: HRP-CVaR base (BIL pinned 25%) + BL posterior, 0.7/0.3 blend."""
    returns_hist = ctx["returns"]
    w_hrp = mf.hrp_cvar_weights_with_fixed(returns_hist, {"BIL": 0.25})
    if P is None:
        return w_hrp
    try:
        w_bl = mf.bl_mv_weights(returns_hist, prior_weights=w_hrp, P=P, Q=Q, obj="Utility")
    except Exception:  # noqa: BLE001 -- BL can fail on degenerate inputs; fall back to base
        return w_hrp
    w = (1.0 - TILT) * w_hrp + TILT * w_bl
    return w / w.sum()


config = steering.SteeringConfig()  # enabled, threshold=0.8, consistency_floor=0.5
print("SteeringConfig:", config)


def _targets_for_agent(agent) -> pd.DataFrame:
    wf = steering.make_steered_weight_fn(
        agent=agent, scorer=prescored, real_symbols=SYMBOLS,
        build_inputs=build_inputs, combine=combine, config=config)
    return mf.build_walk_forward_targets(
        prices[SYMBOLS], rebalance_dates=rebalance_dates,
        weight_fns={"v": wf}, macro_panel=macro_panel, lookback_days=LOOKBACK_DAYS)["v"]


targets_v1 = _targets_for_agent(v1_agent)
targets_v2 = _targets_for_agent(v2_lookup_agent)
print("v1 target rows:", targets_v1.dropna(how="all").shape[0],
      "| v2 target rows:", targets_v2.dropna(how="all").shape[0])

SteeringConfig: SteeringConfig(enabled=True, threshold=0.8, consistency_floor=0.5)


v1 target rows: 72 | v2 target rows: 72


## 9. Head-to-head deltas v2 - v1 (R4.3) + accept-gate (R4.5)

Each version's targets are simulated on the same prices (vbt). We build a head-to-head
table with both versions (plus Baseline / Track B for context) via the unchanged
`mf.head_to_head_report`, then compute the load-bearing v2 - v1 metric deltas and apply
the accept-gate: adopt v2 only if its head-to-head metrics are no worse than v1's.
"No worse" = higher-is-better metrics (return/Sharpe/Sortino/Calmar) not materially lower
and lower-is-better metrics (vol / max drawdown / turnover) not materially higher, within a
small tolerance.


In [10]:
pf_v1 = mf.run_rebalance_sim(prices[SYMBOLS], targets_v1, init_cash=INIT_CASH)
pf_v2 = mf.run_rebalance_sim(prices[SYMBOLS], targets_v2, init_cash=INIT_CASH)

tgt_baseline = pd.read_parquet(DATA / "baseline_targets_2019_2024.parquet")
tgt_track_b = pd.read_parquet(DATA / "track_b_targets_2019_2024.parquet")

pfs = {
    "Baseline":            mf.run_rebalance_sim(prices[SYMBOLS], tgt_baseline, init_cash=INIT_CASH),
    "Track B (MC-Nash)":   mf.run_rebalance_sim(prices[SYMBOLS], tgt_track_b, init_cash=INIT_CASH),
    "Track A v1 (steered)": pf_v1,
    "Track A v2 (steered)": pf_v2,
}
targets = {
    "Baseline": tgt_baseline,
    "Track B (MC-Nash)": tgt_track_b,
    "Track A v1 (steered)": targets_v1,
    "Track A v2 (steered)": targets_v2,
}

report = mf.head_to_head_report(pfs, targets, crisis_start="2022-01-01", crisis_end="2022-12-31")
report.round(4)

,total_return,annualized_return,annualized_vol,sharpe,sortino,calmar,max_drawdown,crisis_return,crisis_max_drawdown,defensive_lead_date,avg_turnover
Baseline,0.158364,0.019945,0.069255,0.319855,0.435338,0.112586,-0.177156,-0.147272,-0.171918,2019-01-02,0.409345
Track B (MC-Nash),0.916609,0.091328,0.115772,0.813043,1.140938,0.370695,-0.24637,-0.113228,-0.161083,2020-11-02,0.163334
Track A v1 (steered),1.126209,0.10665,0.084578,1.240597,1.826633,0.819703,-0.130108,-0.051317,-0.130108,2019-01-02,0.097305
Track A v2 (steered),1.027601,0.099613,0.083231,1.182659,1.728035,0.799537,-0.124588,-0.046014,-0.124588,2019-01-02,0.12109


In [11]:
# Load-bearing head-to-head deltas: v2 - v1 (R4.3).
higher_better = ["total_return", "annualized_return", "sharpe", "sortino", "calmar",
                 "crisis_return"]
lower_better = ["annualized_vol", "max_drawdown", "crisis_max_drawdown", "avg_turnover"]
metric_cols = higher_better + lower_better

v2_row = report.loc["Track A v2 (steered)", metric_cols].astype(float)
v1_row = report.loc["Track A v1 (steered)", metric_cols].astype(float)
delta = (v2_row - v1_row).round(6)
print("Head-to-head delta (v2 - v1):")
for k in metric_cols:
    direction = "higher=better" if k in higher_better else "lower=better"
    print(f"  {k:>20} = {delta[k]:+.6f}   ({direction})")

Head-to-head delta (v2 - v1):
          total_return = -0.098609   (higher=better)
     annualized_return = -0.007037   (higher=better)
                sharpe = -0.057938   (higher=better)
               sortino = -0.098598   (higher=better)
                calmar = -0.020166   (higher=better)
         crisis_return = +0.005303   (higher=better)
        annualized_vol = -0.001347   (lower=better)
          max_drawdown = +0.005520   (lower=better)
   crisis_max_drawdown = +0.005520   (lower=better)
          avg_turnover = +0.023784   (lower=better)


## 10. Accept-gate decision (R4.5, R5.3) — non-predictive

The gate adopts v2 only when it does not degrade the head-to-head evaluation. We use a
small absolute tolerance so floating / sim noise does not flip the decision. Success is
lower-or-equal measured contamination with non-degraded head-to-head metrics — NOT
improved forecast accuracy (R5.3): the gate never rewards forecasting skill; it only
refuses to adopt a refinement that makes the existing evaluation worse.


In [12]:
TOL = 1e-3  # tolerance: changes within +/- TOL count as "not worse"

degraded = []
for k in higher_better:
    if delta[k] < -TOL:
        degraded.append((k, float(delta[k]), "dropped"))
for k in lower_better:
    if delta[k] > TOL:
        degraded.append((k, float(delta[k]), "rose"))

h2h_not_worse = len(degraded) == 0

# Contamination comparison (same shared input stream -> same distribution; reported per
# version for completeness). v2's intent is lower-or-equal contamination.
pm_v1 = report_v1["p_mem_mean"]
pm_v2 = report_v2["p_mem_mean"]
contamination_not_worse = pm_v2 <= pm_v1 + 1e-9

ACCEPT_V2 = h2h_not_worse  # the binding gate is "head-to-head no worse" (R4.5)

print("=== Accept-gate decision (R4.5 / R5.3) ===")
print(f"v1 p_memorized mean = {pm_v1:.4f} | v2 p_memorized mean = {pm_v2:.4f}  "
      f"(contamination not worse: {contamination_not_worse})")
print(f"head-to-head no worse than v1 (tol={TOL}): {h2h_not_worse}")
if degraded:
    print("  degraded metrics (block adoption):")
    for k, d, how in degraded:
        print(f"    {k}: {d:+.6f} ({how})")
else:
    print("  no head-to-head metric degraded beyond tolerance.")
print()
print(f"DECISION: {'ACCEPT v2' if ACCEPT_V2 else 'REJECT v2 (keep v1)'}")
print("Both versions are preserved regardless (distinct caches + artifact files; R4.4).")
print("Success is non-predictive (R5.3): lower-or-equal contamination + non-degraded")
print("head-to-head — never forecast accuracy.")

=== Accept-gate decision (R4.5 / R5.3) ===
v1 p_memorized mean = 0.1513 | v2 p_memorized mean = 0.1513  (contamination not worse: True)
head-to-head no worse than v1 (tol=0.001): False
  degraded metrics (block adoption):
    total_return: -0.098609 (dropped)
    annualized_return: -0.007037 (dropped)
    sharpe: -0.057938 (dropped)
    sortino: -0.098598 (dropped)
    calmar: -0.020166 (dropped)
    max_drawdown: +0.005520 (rose)
    crisis_max_drawdown: +0.005520 (rose)
    avg_turnover: +0.023784 (rose)

DECISION: REJECT v2 (keep v1)
Both versions are preserved regardless (distinct caches + artifact files; R4.4).
Success is non-predictive (R5.3): lower-or-equal contamination + non-degraded
head-to-head — never forecast accuracy.


## 11. Persist per-version artifacts (R4.4, R6.4) — versioned + additive

We persist, per version, the price-independent views + score-distribution logs
(committable) plus a top-level comparison artifact recording the deltas + the accept/reject
decision. The price-dependent targets/equity are written under new versioned filenames
but are gitignored by the parent (they depend on the yfinance price source, like nb11's
steered artifacts). Nothing existing is overwritten; prior versions are preserved.


In [13]:
def _serialize_views_log(views_log: dict) -> dict:
    return {str(k): v for k, v in views_log.items()}


common_meta = {
    "nim_model": NIM_MODEL, "cutoff_date": CUTOFF.isoformat(),
    "holdout_auc": float(scorer.holdout_auc), "is_weak": bool(scorer.is_weak),
    "config": {"enabled": config.enabled, "threshold": config.threshold,
               "consistency_floor": config.consistency_floor},
    "n_rebalances": len(prompt_stream),
}

# Per-version price-INDEPENDENT logs (committable).
v1_log_path = DATA / "prompt_refinement_v1_scores.json"
v2_log_path = DATA / "prompt_refinement_v2_scores.json"
v1_log_path.write_text(json.dumps({
    "meta": {**common_meta, "prompt_version": "v1", "prompt": AGENT_INSTRUCTIONS},
    "p_memorized_distribution": report_v1,
    "view_stability": stab_v1,
    "views": _serialize_views_log(views_log_v1),
}, indent=2, default=str))
v2_log_path.write_text(json.dumps({
    "meta": {**common_meta, "prompt_version": "v2", "prompt": PROMPT_V2},
    "p_memorized_distribution": report_v2,
    "view_stability": stab_v2,
    "views": _serialize_views_log(views_log_v2),
}, indent=2, default=str))

# Top-level comparison + decision (committable).
comparison_path = DATA / "prompt_refinement_comparison.json"
comparison_path.write_text(json.dumps({
    "meta": common_meta,
    "p_memorized": {"v1_mean": float(pm_v1), "v2_mean": float(pm_v2),
                    "contamination_not_worse": bool(contamination_not_worse)},
    "view_stability": {"v1": stab_v1, "v2": stab_v2},
    "head_to_head_delta_v2_minus_v1": {k: float(delta[k]) for k in metric_cols},
    "accept_gate": {"tolerance": TOL, "head_to_head_not_worse": bool(h2h_not_worse),
                    "degraded_metrics": [{"metric": k, "delta": d, "how": how} for k, d, how in degraded],
                    "decision": "ACCEPT_V2" if ACCEPT_V2 else "REJECT_V2_KEEP_V1"},
}, indent=2, default=str))

# Per-version price-DEPENDENT targets/equity (new filenames; gitignored by parent).
v1_targets_path = DATA / "prompt_refinement_v1_targets_2019_2024.parquet"
v2_targets_path = DATA / "prompt_refinement_v2_targets_2019_2024.parquet"
v1_equity_path = DATA / "prompt_refinement_v1_equity_2019_2024.parquet"
v2_equity_path = DATA / "prompt_refinement_v2_equity_2019_2024.parquet"
targets_v1.to_parquet(v1_targets_path)
targets_v2.to_parquet(v2_targets_path)
pf_v1.value().to_frame("value").to_parquet(v1_equity_path)
pf_v2.value().to_frame("value").to_parquet(v2_equity_path)

for p in (v1_log_path, v2_log_path, comparison_path, v1_targets_path, v2_targets_path,
          v1_equity_path, v2_equity_path):
    print("wrote", p.name, "->", p.exists())

wrote prompt_refinement_v1_scores.json -> True
wrote prompt_refinement_v2_scores.json -> True
wrote prompt_refinement_comparison.json -> True
wrote prompt_refinement_v1_targets_2019_2024.parquet -> True
wrote prompt_refinement_v2_targets_2019_2024.parquet -> True
wrote prompt_refinement_v1_equity_2019_2024.parquet -> True
wrote prompt_refinement_v2_equity_2019_2024.parquet -> True


## 12. Conclusion — non-predictive prompt refinement (R5.3)

This playbook compared two prompt versions over the same PIT prompt stream: v1 (the
baseline `AGENT_INSTRUCTIONS`, replayed) and v2 (a refined prompt that reasons explicitly
from the z-scored macro statistics and avoids event recall, run live through
`VariantMacroAgent`). For each version it reported the `p_memorized` distribution (with
`parse_fail_rate`), view-stability, and the head-to-head deltas v2 - v1, then applied
an accept-gate that adopts v2 only when its head-to-head metrics are no worse than
v1's. Both versions are preserved (distinct caches + versioned artifacts).

Success is non-predictive (R5.3): a refinement is acceptable only when it shows
lower-or-equal measured contamination with non-degraded head-to-head metrics, never
because it forecasts returns better. The cell below restates the verdict against that
definition.

> Coverage caveat (honest, not a bug). `meta/llama-4-maverick-17b-128e-instruct`
> frequently misses recall_guard's strict `Direction:/Confidence:` directional format on
> the dateless/anonymized macro prompts, so `parse_fail_rate` is typically high (~0.8) and
> only the parse-OK subset carries a measured `p_memorized`; rebalances without a valid
> score degrade gracefully to unsteered (graceful-degradation contract, R1.6),
> surfaced via `parse_fail_rate` per version. A more format-reliable scoring model or a
> more constrained template would lift coverage; the low-contamination, non-degraded
> conclusion holds on the scored subset.

> Environment caveat. Prices are fetched via `yfinance` (the Postgres `etf_prices` DB
> is unprovisioned here); both versions and all baselines use the SAME yfinance prices, so
> the v2-vs-v1 comparison is internally consistent. The price-dependent targets/equity
> artifacts are gitignored (price-source-dependent); the price-independent views +
> score-distribution + comparison logs are the committable deliverable.


In [14]:
print("=== nb12 prompt-refinement summary (non-predictive, R5.3) ===")
print(f"Calibrator: holdout_auc={scorer.holdout_auc:.3f}, is_weak={scorer.is_weak} -> steering ENABLED")
print()
print(f"v1 (baseline)  p_mem mean={report_v1['p_mem_mean']:.4f}  "
      f"parse_fail_rate={report_v1['parse_fail_rate']:.3f}  "
      f"view_stability(long_switch_rate)={stab_v1['long_switch_rate']:.3f}")
print(f"v2 (refined)   p_mem mean={report_v2['p_mem_mean']:.4f}  "
      f"parse_fail_rate={report_v2['parse_fail_rate']:.3f}  "
      f"view_stability(long_switch_rate)={stab_v2['long_switch_rate']:.3f}")
print()
print("Head-to-head v2 - v1 (key metrics):")
for k in ("total_return", "sharpe", "sortino", "max_drawdown", "avg_turnover"):
    print(f"  {k:>16} = {delta[k]:+.6f}")
print()
print(f"ACCEPT-GATE DECISION: {'ACCEPT v2' if ACCEPT_V2 else 'REJECT v2 (keep v1)'}  "
      f"(head-to-head no worse than v1: {h2h_not_worse})")
print("Both prompt versions preserved (distinct caches + versioned artifacts; R4.4).")
print("Verdict basis = lower-or-equal contamination + non-degraded head-to-head, NOT forecast accuracy.")

=== nb12 prompt-refinement summary (non-predictive, R5.3) ===
Calibrator: holdout_auc=0.988, is_weak=False -> steering ENABLED

v1 (baseline)  p_mem mean=0.1513  parse_fail_rate=0.792  view_stability(long_switch_rate)=0.310
v2 (refined)   p_mem mean=0.1513  parse_fail_rate=0.792  view_stability(long_switch_rate)=0.324

Head-to-head v2 - v1 (key metrics):
      total_return = -0.098609
            sharpe = -0.057938
           sortino = -0.098598
      max_drawdown = +0.005520
      avg_turnover = +0.023784

ACCEPT-GATE DECISION: REJECT v2 (keep v1)  (head-to-head no worse than v1: False)
Both prompt versions preserved (distinct caches + versioned artifacts; R4.4).
Verdict basis = lower-or-equal contamination + non-degraded head-to-head, NOT forecast accuracy.
